# SQL for Data Cleaning, Joining, and Window Functions

> **Level:** Beginner (you should know what a table is and have run a few `SELECT` queries) | **Code language:** Python (with SQL inside strings) | **Estimated study time:** 5 to 7 hours

## Table of Contents

- [Phase 1: Getting Data Out of a Table](#phase-1-getting-data-out-of-a-table)
  - [Lesson 1.1: SELECT and FROM](#lesson-11-select-and-from)
  - [Lesson 1.2: Filtering rows with WHERE](#lesson-12-filtering-rows-with-where)
  - [Lesson 1.3: Sorting and limiting](#lesson-13-sorting-and-limiting)
  - [Lesson 1.4: Aliases and DISTINCT](#lesson-14-aliases-and-distinct)
- [Phase 2: Cleaning Messy Values](#phase-2-cleaning-messy-values)
  - [Lesson 2.1: Bracket-quoting odd column names](#lesson-21-bracket-quoting-odd-column-names)
  - [Lesson 2.2: Removing text with REPLACE](#lesson-22-removing-text-with-replace)
  - [Lesson 2.3: Changing types with CAST and TRY_CAST](#lesson-23-changing-types-with-cast-and-try_cast)
  - [Lesson 2.4: Trimming and tidying text](#lesson-24-trimming-and-tidying-text)
- [Phase 3: Working with Dates](#phase-3-working-with-dates)
  - [Lesson 3.1: ISO dates and BETWEEN](#lesson-31-iso-dates-and-between)
  - [Lesson 3.2: Converting text to dates](#lesson-32-converting-text-to-dates)
  - [Lesson 3.3: Extracting parts of a date](#lesson-33-extracting-parts-of-a-date)
- [Phase 4: Summarising Data](#phase-4-summarising-data)
  - [Lesson 4.1: COUNT, SUM, AVG, MIN, MAX](#lesson-41-count-sum-avg-min-max)
  - [Lesson 4.2: GROUP BY](#lesson-42-group-by)
  - [Lesson 4.3: HAVING](#lesson-43-having)
  - [Lesson 4.4: ROUND](#lesson-44-round)
- [Phase 5: Joining Tables](#phase-5-joining-tables)
  - [Lesson 5.1: INNER JOIN](#lesson-51-inner-join)
  - [Lesson 5.2: LEFT JOIN and finding non-matches](#lesson-52-left-join-and-finding-non-matches)
  - [Lesson 5.3: Joining three tables](#lesson-53-joining-three-tables)
- [Phase 6: CTEs and Window Functions](#phase-6-ctes-and-window-functions)
  - [Lesson 6.1: WITH (the CTE)](#lesson-61-with-the-cte)
  - [Lesson 6.2: SUM OVER and ROWS BETWEEN](#lesson-62-sum-over-and-rows-between)
  - [Lesson 6.3: A rolling 7-day total](#lesson-63-a-rolling-7-day-total)
  - [Lesson 6.4: ROW_NUMBER and RANK](#lesson-64-row_number-and-rank)
- [Glossary](#glossary)
- [Answer Key](#answer-key)

## How to Use This Note

- Read each lesson in order. Do not skip ahead.
- Before you run a SQL cell, guess the result.
- Run every cell yourself. The cells use a real in-memory database built inside the notebook, so they run without any external file or server.
- Do the Practice Task at the end of every Phase. It uses only skills taught up to that point.
- If a query fails, read the error message slowly. Most errors come from a typo in a column name or a missing bracket.

---

## Phase 1: Getting Data Out of a Table

**Phase goal:** Read rows and columns from a table using `SELECT`, filter them with `WHERE`, sort them, and pick only the rows you want.

**You need before this phase:** Nothing. This is the starting point.

#### Setup Cell

In [ ]:
# sqlite3 is built into Python. It gives us a real, tiny SQL database
# that lives in memory, so every cell in this notebook runs.
import sqlite3
import pandas as pd

# A fresh database in memory. It is wiped when the notebook restarts.
conn = sqlite3.connect(':memory:')

# This is our stand-in for the cohort's run_query function.
# It runs a SQL query and returns a pandas DataFrame.
def run_query(sql):
    return pd.read_sql_query(sql, conn)

# Create the two sample tables and fill them with rows.
conn.executescript("""
CREATE TABLE london_2014 (
    GMT TEXT,
    [Max TemperatureC] INTEGER,
    [Min TemperatureC] INTEGER,
    Precipitationmm REAL,
    [WindDirDegrees<br />] TEXT
);

INSERT INTO london_2014 VALUES
('2014-01-01', 8, 4, 0.0, '186<br />'),
('2014-01-02', 9, 5, 2.5, '270<br />'),
('2014-01-03', 7, 3, 0.0, '90<br />'),
('2014-01-04', 10, 6, 1.0, '355<br />'),
('2014-01-05', 11, 7, 0.0, '10<br />'),
('2014-01-06', 10, 6, 0.5, '150<br />'),
('2014-01-07', 9, 5, 0.0, '160<br />'),
('2014-01-08', 8, 4, 0.0, '170<br />'),
('2014-01-09', 7, 3, 0.0, '180<br />'),
('2014-01-10', 6, 2, 0.5, '190<br />'),
('2014-06-01', 18, 12, 0.0, '270<br />'),
('2014-06-02', 19, 13, 0.0, '270<br />'),
('2014-06-03', 20, 14, 0.0, '280<br />'),
('2014-06-04', 21, 15, 0.5, '290<br />'),
('2014-06-05', 20, 14, 0.0, '270<br />'),
('2014-06-06', 19, 13, 0.0, '260<br />'),
('2014-06-07', 18, 12, 0.0, '250<br />'),
('2014-06-08', 19, 13, 1.0, '240<br />'),
('2014-06-09', 20, 14, 0.0, '230<br />'),
('2014-06-10', 21, 15, 0.0, '220<br />'),
('2014-12-08', 6, 2, 0.5, '180<br />'),
('2014-12-09', 5, 1, 3.0, '200<br />'),
('2014-12-10', 7, 3, 0.0, '220<br />'),
('2014-12-11', 9, 4, 1.5, '250<br />'),
('2014-12-12', 8, 3, 0.0, '270<br />');

CREATE TABLE wb_gdp_2013 (
    country TEXT,
    [NY.GDP.MKTP.CD] REAL
);

INSERT INTO wb_gdp_2013 VALUES
('Afghanistan', 20458940000),
('Albania', 12781030000),
('Algeria', 209703500000),
('Angola', 138356800000),
('India', 1876787989000);

CREATE TABLE wb_le_2013 (
    country TEXT,
    [SP.DYN.LE00.IN] REAL
);

INSERT INTO wb_le_2013 VALUES
('Afghanistan', 60.3),
('Albania', 77.8),
('Algeria', 75.1),
('Angola', 52.4),
('Nigeria', 55.0);

CREATE TABLE wb_pop_2013 (
    country TEXT,
    [SP.POP.TOTL] REAL
);

INSERT INTO wb_pop_2013 VALUES
('Afghanistan', 30552000),
('Albania', 2895000),
('Algeria', 38282000),
('Angola', 25789000);
""")

print("Database ready. Tables: london_2014, wb_gdp_2013, wb_le_2013, wb_pop_2013")




### Lesson 1.1: SELECT and FROM

#### What You Will Learn
- What `SELECT` does.
- What `FROM` does.
- How `SELECT *` differs from naming columns.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **Table** | A grid of rows and columns stored in a database. |
| **Row** | One record in a table. |
| **Column** | One field that every row shares. |
| **SELECT** | The SQL keyword that names the columns you want. |
| **FROM** | The SQL keyword that names the table you want. |

#### The Idea in Plain Words

Imagine a shoe shop. The shop is the database. The shelves are tables. Each box on a shelf is a row. Each label on a box is a column.

You walk up to the counter and say: "Give me these columns from that shelf." In SQL, you write `SELECT <columns> FROM <table>`. The database goes to the shelf and brings you a new, short table with only the columns you asked for.

If you say "give me everything", you write `SELECT *`. The asterisk means "every column".

#### Tool Card: sqlite3
- **What it is:** A small SQL database engine built into Python.
- **What it does:** Stores tables in memory or in a file, and answers SQL queries.
- **Why we use it here:** It lets you run real SQL in a notebook with no server to install.
- **Install:** Built in, no install needed.
- **Import:** `import sqlite3`
- **Other options:** DuckDB (fast, similar to SQLite), PostgreSQL (large, production), SQL Server (what the cohort uses).

> **Dialect note:** The cohort uses Microsoft SQL Server. SQLite and SQL Server share most of the SQL language. Where they differ, this note shows both.

#### See It in Code

Before you run this, guess how many columns the result has.

In [ ]:
# Ask for two specific columns from the weather table.
result = run_query("""
    SELECT GMT, [Max TemperatureC]
    FROM london_2014
""")
print(result.head())




**Expected output:**
```text
          GMT  Max TemperatureC
0  2014-01-01                 8
1  2014-01-02                 9
2  2014-01-03                 7
3  2014-01-04                10
4  2014-01-05                11
```

**What happened:** The query returned every row of `london_2014`, but only the two columns you asked for.

#### Common Mistakes

```text
OperationalError: no such table: london_2014
```

The setup cell was not run. Fix: run the Setup Cell first.

```text
OperationalError: no such column: Max TemperatureC
```

The column name has a space. Fix: wrap it in square brackets: `[Max TemperatureC]`.

#### Quick Check
1. What does `SELECT *` mean?
2. What does `FROM` do?
3. Why does `[Max TemperatureC]` need square brackets?

#### Lesson Recap
- `SELECT` names the columns.
- `FROM` names the table.
- `SELECT *` means "every column".
- Spaces in a column name need square brackets.

---

### Lesson 1.2: Filtering rows with WHERE

#### What You Will Learn
- How to keep only some rows with `WHERE`.
- How to combine conditions with `AND` and `OR`.
- How to use comparison operators: `=`, `<`, `>`, `<=`, `>=`, `<>`.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **WHERE** | The SQL keyword that keeps only rows that match a condition. |
| **Condition** | A test that is true or false for each row. |
| **AND** | Both conditions must be true. |
| **OR** | At least one condition must be true. |
| **<>** | "Not equal to". Some databases also accept `!=`. |

#### The Idea in Plain Words

Imagine you are sorting a basket of apples. `WHERE colour = 'red'` is like picking up only red apples. `WHERE weight > 100 AND colour = 'red'` is like picking up only red apples that also weigh more than 100 grams.

`WHERE` runs on every row. If the test is true, the row stays. If false, the row is dropped.

#### See It in Code

Before you run this, guess how many rows match.

In [ ]:
result = run_query("""
    SELECT GMT, [Max TemperatureC]
    FROM london_2014
    WHERE [Max TemperatureC] >= 10
""")
print(result)




**Expected output:**
```text
          GMT  Max TemperatureC
0  2014-01-04                10
1  2014-01-05                11
2  2014-01-06                10
3  2014-06-04                21
4  2014-06-05                20
5  2014-06-09                20
6  2014-06-10                21
```

**What happened:** Only rows where the maximum temperature was 10 or more were kept. Seven rows matched.

#### Common Mistakes

```text
OperationalError: near "WHERE": syntax error
```

You wrote `WHERE` before `FROM`. Fix: `FROM` comes before `WHERE`.

#### Quick Check
1. Which rows does `WHERE [Max TemperatureC] >= 10` keep?
2. What does `AND` do to two conditions?
3. Which operator means "not equal to"?

#### Lesson Recap
- `WHERE` keeps rows that match a condition.
- `AND` and `OR` combine conditions.
- `<>` means "not equal to".

---

### Lesson 1.3: Sorting and limiting

#### What You Will Learn
- How to sort with `ORDER BY`.
- How to sort ascending vs descending.
- How to keep only the top N rows with `LIMIT` (or `TOP` in SQL Server).

#### Key Terms
| Term | Simple meaning |
|---|---|
| **ORDER BY** | Sorts the result. |
| **ASC** | Ascending (smallest first). The default. |
| **DESC** | Descending (largest first). |
| **LIMIT** | Keeps only the first N rows after sorting. |
| **TOP** | The SQL Server version of `LIMIT`, written before the columns. |

#### The Idea in Plain Words

A leaderboard is a sorted list. `ORDER BY score DESC` puts the highest score at the top. `LIMIT 3` keeps only the top three. Without `LIMIT`, you get every row, sorted.

#### See It in Code

Before you run this, guess which date appears first.

In [ ]:
result = run_query("""
    SELECT GMT, [Max TemperatureC]
    FROM london_2014
    ORDER BY [Max TemperatureC] DESC, GMT ASC
    LIMIT 5
""")
print(result)




**Expected output:**
```text
          GMT  Max TemperatureC
0  2014-06-04                21
1  2014-06-10                21
2  2014-06-05                20
3  2014-06-09                20
4  2014-01-05                11
```

**What happened:** The table was sorted by temperature (highest first). When two rows had the same temperature, the earlier date came first. Only five rows were kept.

#### Common Mistakes

```text
OperationalError: near "TOP": syntax error
```

You used `TOP` but the database is SQLite. Fix: use `LIMIT` at the end. SQL Server uses `SELECT TOP 5 ...`.

#### Quick Check
1. Which order does `ORDER BY x DESC` use?
2. Where does `LIMIT` go in the query?
3. In SQL Server, where does `TOP` go?

#### Lesson Recap
- `ORDER BY` sorts the result.
- `DESC` puts the largest value first.
- `LIMIT` keeps the first N rows after sorting.

---

### Lesson 1.4: Aliases and DISTINCT

#### What You Will Learn
- How to rename a column in the output with `AS`.
- How to remove duplicate rows with `DISTINCT`.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **AS** | Renames a column in the output. |
| **Alias** | The new name you give a column or table. |
| **DISTINCT** | Removes duplicate rows from the result. |

#### The Idea in Plain Words

An alias is a nickname. If a column is called `[NY.GDP.MKTP.CD]`, you can show it as `gdp` instead. `DISTINCT` is like sorting a bag of marbles and keeping only one of each colour.

#### See It in Code

Before you run this, guess how many rows the result has.

In [ ]:
result = run_query("""
    SELECT country AS name, [NY.GDP.MKTP.CD] AS gdp_usd
    FROM wb_gdp_2013
    ORDER BY gdp_usd DESC
""")
print(result)




**Expected output:**
```text
           name         gdp_usd
0         India  1.876788e+12
1       Algeria  2.097035e+11
2        Angola  1.383568e+11
3   Afghanistan  2.045894e+10
4       Albania  1.278103e+10
```

**What happened:** `country` was shown as `name`. The GDP column was shown as `gdp_usd`. The rows were sorted from richest to poorest.

#### Common Mistakes

```text
OperationalError: near "AS": syntax error
```

You used `AS` after `WHERE` or in another clause. Fix: aliases go in the `SELECT` clause.

#### Quick Check
1. What does `AS` do?
2. What does `DISTINCT` remove?
3. What is an alias?

#### Lesson Recap
- `AS` gives a column a new name in the output.
- `DISTINCT` removes duplicate rows.
- Aliases keep long column names out of your result.

### Phase 1 Summary
- You can read a whole table or selected columns.
- You can filter rows with `WHERE`.
- You can sort with `ORDER BY` and keep the top N with `LIMIT`.
- You can rename columns with `AS`.

### Phase 1 Practice Task: Read the weather for a chosen month

**Difficulty:** Easy | **Time:** 15 minutes

**Skills used:**
- From this phase: `SELECT` and `FROM` (Lesson 1.1), `WHERE` (Lesson 1.2), `ORDER BY` and `LIMIT` (Lesson 1.3), aliases (Lesson 1.4)
- From earlier phases: None. This is your first task.

**The task:** Write one SQL query against `london_2014`. Return every row from January 2014 (dates between `'2014-01-01'` and `'2014-01-31'`, inclusive). Show the columns `GMT`, `[Max TemperatureC]`, `[Min TemperatureC]`. Sort by `GMT` ascending. Rename `GMT` to `day` and `[Max TemperatureC]` to `t_max` in the output.

The table has these January rows (all in `london_2014`):

| GMT | Max TemperatureC | Min TemperatureC |
|---|---|---|
| 2014-01-01 | 8 | 4 |
| 2014-01-02 | 9 | 5 |
| 2014-01-03 | 7 | 3 |
| 2014-01-04 | 10 | 6 |
| 2014-01-05 | 11 | 7 |
| 2014-01-06 | 10 | 6 |
| 2014-01-07 | 9 | 5 |
| 2014-01-08 | 8 | 4 |
| 2014-01-09 | 7 | 3 |
| 2014-01-10 | 6 | 2 |

**Expected output:**
```text
         day  t_max  Min TemperatureC
0 2014-01-01      8                4
1 2014-01-02      9                5
2 2014-01-03      7                3
3 2014-01-04     10                6
4 2014-01-05     11                7
5 2014-01-06     10                6
6 2014-01-07      9                5
7 2014-01-08      8                4
8 2014-01-09      7                3
9 2014-01-10      6                2
```

---

## Phase 2: Cleaning Messy Values

**Phase goal:** Remove unwanted characters and change data types directly inside SQL.

**You need before this phase:** Phase 1.

#### Setup Cell

In [ ]:
# A fresh start. The tables were built in Phase 1 setup.
# If you restarted the notebook, run the Phase 1 setup cell first.

# Confirm the column name has the messy suffix.
result = run_query("SELECT * FROM london_2014 LIMIT 1")
print(result.columns.tolist())




### Lesson 2.1: Bracket-quoting odd column names

#### What You Will Learn
- Why some column names need square brackets.
- How to select a bracketed column.
- Why this matters for names with spaces, dots, or HTML tags.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **Bracket-quote** | Wrap a name in `[ ... ]` so SQL reads it as one name. |
| **Identifier** | The name of a table or column. |
| **Reserved word** | A word SQL treats specially, like `SELECT` or `FROM`. |

#### The Idea in Plain Words

Some names contain characters that SQL would otherwise misread. A space, a dot, or a `<` symbol breaks the name into pieces. Square brackets say "everything inside is the name".

Compare a name tag. If your name is "Smith", a plain tag works. If your name is "Smith & Sons", you need quotes so the tag reader does not treat "&" as a symbol.

#### See It in Code

Before you run this, guess which of the two queries fails.

In [ ]:
# This works: the name is inside brackets.
result = run_query("""
    SELECT [WindDirDegrees<br />]
    FROM london_2014
    LIMIT 3
""")
print(result)




**Expected output:**
```text
  WindDirDegrees<br />
0            186<br />
1            270<br />
2             90<br />
```

**What happened:** The bracketed name was read as one column. Its values still carry the `<br />` tag.

#### Common Mistakes

```text
OperationalError: near "<": syntax error
```

You wrote `WindDirDegrees<br />` without brackets. Fix: wrap the whole name in `[ ... ]`.

#### Quick Check
1. When does a column name need brackets?
2. What does a reserved word do in SQL?
3. Does the bracket change the column name or only the way you type it?

#### Lesson Recap
- Bracket-quote names with spaces, dots, or symbols.
- `[WindDirDegrees<br />]` is one column name.
- The brackets are not part of the name.

---

### Lesson 2.2: Removing text with REPLACE

#### What You Will Learn
- How `REPLACE` swaps one piece of text for another.
- How to strip `<br />` from every value in a column.
- How to count rows that meet a cleaned condition.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **REPLACE(text, find, replace)** | Returns text with every `find` swapped for `replace`. |

#### The Idea in Plain Words

`REPLACE` is like a find-and-replace in a word processor. You give it three things: the original text, what to look for, and what to put in its place. If you set the "replace" part to the empty string `''`, the text is deleted.

#### See It in Code

Before you run this, guess the count.

In [ ]:
# Remove the <br /> tag from each value, then keep values above 350.
# CAST turns the text into an integer.
result = run_query("""
    SELECT COUNT(*) AS days_over_350
    FROM london_2014
    WHERE CAST(REPLACE([WindDirDegrees<br />], '<br />', '') AS INTEGER) > 350
""")
print(result)




**Expected output:**
```text
   days_over_350
0              1
```

**What happened:** `REPLACE` removed the `<br />` from each value. `CAST` turned the text into an integer. Only one day had a wind direction above 350 degrees.

> **Dialect note:** In SQL Server, use `TRY_CAST(... AS int)` instead of `CAST`. `TRY_CAST` returns NULL on bad data instead of raising an error.

#### Common Mistakes

```text
OperationalError: no such column: WindDirDegrees
```

You forgot the brackets. Fix: write `[WindDirDegrees<br />]`.

```text
ValueError: invalid literal
```

You used a database that raises on `CAST` of bad text. Fix: use `TRY_CAST` on SQL Server, or clean with `REPLACE` first.

#### Quick Check
1. What does `REPLACE(text, '<br />', '')` do?
2. What does `COUNT(*)` return?
3. Why do we cast to `INTEGER` before comparing with `> 350`?

#### Lesson Recap
- `REPLACE` removes or swaps text.
- `CAST` changes a value's type.
- `TRY_CAST` does the same without raising on bad data.

---

### Lesson 2.3: Changing types with CAST and TRY_CAST

#### What You Will Learn
- What `CAST` does and when it fails.
- How `TRY_CAST` avoids errors.
- How the two differ across dialects.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **CAST** | Change a value to a different type. Raises an error if it cannot. |
| **TRY_CAST** | Change a value to a different type. Returns NULL on failure. |
| **INTEGER** | Whole numbers. |
| **REAL** | Decimal numbers. |
| **TEXT** | Text values. |

#### The Idea in Plain Words

A price written on a paper slip is text. A number on a calculator is a number. `CAST('186' AS INTEGER)` turns the text `'186'` into the number 186. But `CAST('186<br />' AS INTEGER)` fails because the extra text is not a number.

`TRY_CAST` is the gentle version. If it cannot convert, it returns `NULL` instead of stopping the query.

#### See It in Code

Before you run this, guess how many rows are NULL.

In [ ]:
# Clean first, then cast. That avoids the <br /> problem.
result = run_query("""
    SELECT GMT,
           CAST(REPLACE([WindDirDegrees<br />], '<br />', '') AS INTEGER) AS wind_dir
    FROM london_2014
    LIMIT 3
""")
print(result)




**Expected output:**
```text
          GMT  wind_dir
0  2014-01-01       186
1  2014-01-02       270
2  2014-01-03        90
```

**What happened:** `REPLACE` removed `<br />`. `CAST` turned the cleaned text into an integer.

> **Version note:** SQL Server uses `TRY_CAST(x AS int)`. PostgreSQL uses `CAST(x AS integer)`. SQLite uses `CAST(x AS INTEGER)`. The name `INTEGER` works in all three.

#### Common Mistakes

```text
OperationalError: near "TRY_CAST": syntax error
```

SQLite does not know `TRY_CAST`. Fix: use `CAST` and clean the data first.

#### Quick Check
1. What does `CAST('186' AS INTEGER)` return?
2. What does `CAST('186<br />' AS INTEGER)` do in SQLite?
3. Why is `TRY_CAST` safer in a messy dataset?

#### Lesson Recap
- `CAST` changes the type.
- `TRY_CAST` changes the type without raising on bad data.
- Clean the data first, then cast.

---

### Lesson 2.4: Trimming and tidying text

#### What You Will Learn
- How `TRIM` removes spaces from both ends.
- How `LTRIM` and `RTRIM` work on one side.
- How `UPPER` and `LOWER` change case.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **TRIM** | Removes spaces from both ends of text. |
| **LTRIM** | Removes spaces from the left end. |
| **RTRIM** | Removes spaces from the right end. |
| **UPPER** | Turns text into uppercase. |
| **LOWER** | Turns text into lowercase. |

#### The Idea in Plain Words

A pencil case has a lid on each end. `TRIM` opens both lids and removes the spaces inside. `LTRIM` opens only the left lid. `RTRIM` opens only the right. `UPPER` and `LOWER` change the case of every letter.

#### See It in Code

Before you run this, guess the cleaned value.

In [ ]:
result = run_query("""
    SELECT TRIM('  hello  ') AS both_ends,
           LTRIM('  hello  ') AS left_only,
           RTRIM('  hello  ') AS right_only,
           UPPER('hello') AS big,
           LOWER('HELLO') AS small
""")
print(result)




**Expected output:**
```text
  both_ends  left_only  right_only   big  small
0    hello     hello        hello   HELLO  hello
```

**What happened:** `TRIM` removed both spaces. `LTRIM` removed the left space. `RTRIM` removed the right space. `UPPER` and `LOWER` changed case.

> **Dialect note:** SQL Server uses `LTRIM` and `RTRIM`. Recent versions also accept `TRIM`. SQLite accepts `TRIM`, `LTRIM`, and `RTRIM`.

#### Common Mistakes

```text
OperationalError: no such function: TRIMX
```

The function does not exist. Fix: use `TRIM`, `LTRIM`, or `RTRIM`.

#### Quick Check
1. What does `TRIM` remove?
2. Which function removes spaces only from the left?
3. Which function changes text to uppercase?

#### Lesson Recap
- `TRIM` removes spaces from both ends.
- `LTRIM` and `RTRIM` work on one end.
- `UPPER` and `LOWER` change case.

### Phase 2 Summary
- You can bracket-quote odd column names.
- You can remove unwanted text with `REPLACE`.
- You can change types with `CAST` or `TRY_CAST`.
- You can tidy text with `TRIM`, `LTRIM`, `RTRIM`, `UPPER`, and `LOWER`.

### Phase 2 Practice Task: Clean the wind column and count

**Difficulty:** Easy-Medium | **Time:** 20 minutes

**Skills used:**
- From this phase: bracket-quoting (Lesson 2.1), `REPLACE` (Lesson 2.2), `CAST` (Lesson 2.3), `TRIM` (Lesson 2.4)
- From earlier phases: `SELECT`, `WHERE`, `COUNT` (Phase 1, Lessons 1.1 and 1.2)

**The task:** The table `london_2014` has a column named `[WindDirDegrees<br />]`. Every value looks like `'186<br />'`. Write two queries.

1. Return a small table showing `GMT` and a cleaned column called `wind_dir` (an integer). Only include the first 5 rows.
2. Return one row, one column called `days_over_350`. Count the days where the cleaned wind direction is greater than 350.

The first five rows are:

| GMT | WindDirDegrees<br /> |
|---|---|
| 2014-01-01 | 186<br /> |
| 2014-01-02 | 270<br /> |
| 2014-01-03 | 90<br /> |
| 2014-01-04 | 355<br /> |
| 2014-01-05 | 10<br /> |

**Expected output:**
```text
Query 1:
          GMT  wind_dir
0  2014-01-01       186
1  2014-01-02       270
2  2014-01-03        90
3  2014-01-04       355
4  2014-01-05        10

Query 2:
   days_over_350
0              1
```

---

## Phase 3: Working with Dates

**Phase goal:** Read, filter, and extract parts of date columns in SQL.

**You need before this phase:** Phase 2.

#### Setup Cell

In [ ]:
# The dates in london_2014 are stored as text in ISO format
# ('YYYY-MM-DD'). That format sorts correctly as plain text,
# which makes WHERE filtering easy.

# Confirm the date format.
result = run_query("SELECT GMT FROM london_2014 LIMIT 3")
print(result)




### Lesson 3.1: ISO dates and BETWEEN

#### What You Will Learn
- What ISO date format is.
- Why ISO dates sort as plain text.
- How `BETWEEN` keeps a range of values.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **ISO date** | A date written as `YYYY-MM-DD`, like `2014-01-01`. |
| **BETWEEN** | Keep values that fall inside a range, both ends included. |

#### The Idea in Plain Words

An ISO date always writes the year first, then the month, then the day. Because every part is padded to the same length (`2014-01-01`, not `2014-1-1`), the text sorts exactly like the dates it represents.

That means `WHERE GMT BETWEEN '2014-12-08' AND '2014-12-12'` works even though `GMT` is stored as text. You do not need a special date type.

#### See It in Code

Before you run this, guess how many rows match.

In [ ]:
result = run_query("""
    SELECT GMT, [Max TemperatureC], [Min TemperatureC]
    FROM london_2014
    WHERE GMT BETWEEN '2014-12-08' AND '2014-12-12'
    ORDER BY GMT
""")
print(result)




**Expected output:**
```text
          GMT  Max TemperatureC  Min TemperatureC
0  2014-12-08                 6                 2
1  2014-12-09                 5                 1
2  2014-12-10                 7                 3
3  2014-12-11                 9                 4
4  2014-12-12                 8                 3
```

**What happened:** Five rows matched the range. Both ends were included. The output was sorted by date.

#### Common Mistakes

```text
No rows returned
```

You compared padded and unpadded dates. Fix: write `'2014-12-08'`, not `'2014-12-8'`.

#### Quick Check
1. What does ISO stand for?
2. Why does `BETWEEN '2014-12-08' AND '2014-12-12'` work on text?
3. Does `BETWEEN` include the end dates?

#### Lesson Recap
- ISO dates sort as text.
- `BETWEEN` keeps a range, both ends included.
- Pad your date strings.

---

### Lesson 3.2: Converting text to dates

#### What You Will Learn
- How to convert a text column to a date with a function.
- How each dialect names that function.
- Why converting can still help even when text comparison works.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **Date type** | A real date value, not text. |
| **Conversion** | Changing a value from one type to another. |

#### The Idea in Plain Words

A string like `'2014-01-01'` is just letters. A date value understands leap years, month lengths, and the difference between two days. Text comparison happens to work for ISO dates, but real date functions (like "give me the weekday" or "add 30 days") need a date type.

#### See It in Code

Before you run this, guess the extracted year.

In [ ]:
# SQLite has a date() function that parses ISO text.
result = run_query("""
    SELECT GMT, date(GMT) AS gmt_date
    FROM london_2014
    LIMIT 3
""")
print(result)




**Expected output:**
```text
          GMT    gmt_date
0  2014-01-01  2014-01-01
1  2014-01-02  2014-01-02
2  2014-01-03  2014-01-03
```

**What happened:** `date(GMT)` returned the same date, but now as a real date value that date functions can use.

> **Dialect note:** SQL Server uses `TRY_CONVERT(date, GMT)`. PostgreSQL uses `CAST(GMT AS date)`. SQLite uses `date(GMT)`.

#### Common Mistakes

```text
OperationalError: no such function: TRY_CONVERT
```

SQLite does not know `TRY_CONVERT`. Fix: use `date(GMT)`.

#### Quick Check
1. What does `date(GMT)` return?
2. Why does text comparison still work for ISO dates?
3. Which function would you use in SQL Server?

#### Lesson Recap
- `date(GMT)` (or `TRY_CONVERT`) turns text into a real date.
- ISO text sorts correctly, so plain `WHERE` works.
- Real date types unlock date functions.

---

### Lesson 3.3: Extracting parts of a date

#### What You Will Learn
- How to pull the year, month, or day out of a date.
- How `strftime` works in SQLite.
- How SQL Server uses `YEAR`, `MONTH`, `DAY`.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **strftime** | A function that formats a date into text. |
| **YEAR** | The year part of a date. |
| **MONTH** | The month part of a date. |
| **DAY** | The day part of a date. |

#### The Idea in Plain Words

A date is like a birthday cake with three layers: year, month, and day. Extracting a part is like cutting one layer. In SQLite you call `strftime('%Y', GMT)` for the year, `'%m'` for the month, `'%d'` for the day.

#### See It in Code

Before you run this, guess the month values.

In [ ]:
result = run_query("""
    SELECT GMT,
           strftime('%Y', date(GMT)) AS year,
           strftime('%m', date(GMT)) AS month,
           strftime('%d', date(GMT)) AS day
    FROM london_2014
    LIMIT 3
""")
print(result)




**Expected output:**
```text
          GMT  year month day
0  2014-01-01  2014    01  01
1  2014-01-02  2014    01  02
2  2014-01-03  2014    01  03
```

**What happened:** The year, month, and day were pulled out of the date as text.

> **Dialect note:** SQL Server uses `YEAR(date)`, `MONTH(date)`, and `DAY(date)`. PostgreSQL uses `EXTRACT(YEAR FROM date)`.

#### Common Mistakes

```text
OperationalError: no such function: YEAR
```

SQLite does not have `YEAR`. Fix: use `strftime('%Y', date(GMT))`.

#### Quick Check
1. Which `strftime` format gives the month?
2. Which function gives the year in SQL Server?
3. What does `strftime` return?

#### Lesson Recap
- `strftime` extracts parts of a date in SQLite.
- SQL Server uses `YEAR`, `MONTH`, `DAY`.
- Extracted parts are text by default.

### Phase 3 Summary
- ISO dates sort as text.
- `BETWEEN` filters a date range.
- Real date types unlock more functions.
- You can extract year, month, and day.

### Phase 3 Practice Task: Winter weather

**Difficulty:** Medium | **Time:** 25 minutes

**Skills used:**
- From this phase: `BETWEEN` (Lesson 3.1), `date()` (Lesson 3.2), `strftime` (Lesson 3.3)
- From earlier phases: `SELECT`, `WHERE`, `ORDER BY` (Phase 1)

**The task:** Write one SQL query against `london_2014`. Return every row where the date is between `'2014-12-08'` and `'2014-12-12'`. Show three columns: `day` (the date as text), `t_max` (the maximum temperature), and `month` (the month number as text). Sort by `day` ascending.

The matching rows are:

| GMT | Max TemperatureC |
|---|---|
| 2014-12-08 | 6 |
| 2014-12-09 | 5 |
| 2014-12-10 | 7 |
| 2014-12-11 | 9 |
| 2014-12-12 | 8 |

**Expected output:**
```text
         day  t_max month
0 2014-12-08      6    12
1 2014-12-09      5    12
2 2014-12-10      7    12
3 2014-12-11      9    12
4 2014-12-12      8    12
```

---

## Phase 4: Summarising Data

**Phase goal:** Collapse many rows into counts, totals, and averages, and group those summaries.

**You need before this phase:** Phase 3.

#### Setup Cell

In [ ]:
# Same tables. Nothing new to create.

# Quick check that the population table is present.
result = run_query("SELECT COUNT(*) AS n FROM wb_pop_2013")
print(result)




### Lesson 4.1: COUNT, SUM, AVG, MIN, MAX

#### What You Will Learn
- What each aggregate function does.
- How the result differs from `SELECT *`.
- How NULL affects aggregates.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **Aggregate function** | A function that collapses many rows into one value. |
| **COUNT** | How many rows. |
| **SUM** | The total of a numeric column. |
| **AVG** | The average of a numeric column. |
| **MIN** | The smallest value. |
| **MAX** | The largest value. |

#### The Idea in Plain Words

Aggregates turn many rows into one number. Imagine a stack of receipts. `SUM` is the total spent. `AVG` is the average per receipt. `COUNT` is how many receipts you have. `MIN` and `MAX` are the cheapest and most expensive.

#### See It in Code

Before you run this, guess each value.

In [ ]:
result = run_query("""
    SELECT COUNT(*) AS rows_count,
           SUM(Precipitationmm) AS total_rain,
           AVG(Precipitationmm) AS avg_rain,
           MIN(Precipitationmm) AS min_rain,
           MAX(Precipitationmm) AS max_rain
    FROM london_2014
""")
print(result)




**Expected output:**
```text
   rows_count  total_rain  avg_rain  min_rain  max_rain
0          25        10.5      0.42       0.0       3.0
```

**What happened:** One row summarised all 25 rows of the table.

> **Plain English:** If some values are NULL, `COUNT(*)` still counts the row, but `AVG` and `SUM` skip the NULLs.

#### Common Mistakes

```text
OperationalError: misuse of aggregate function SUM()
```

You mixed an aggregate with a plain column without `GROUP BY`. Fix: either remove the plain column or add `GROUP BY` (Lesson 4.2).

#### Quick Check
1. What does `COUNT(*)` return?
2. Which aggregate gives the smallest value?
3. Does `AVG` include NULL values?

#### Lesson Recap
- Aggregates collapse many rows into one.
- `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` cover most needs.
- NULL values are skipped by `SUM` and `AVG`.

---

### Lesson 4.2: GROUP BY

#### What You Will Learn
- How to split rows into groups with `GROUP BY`.
- How the aggregate runs per group.
- How `GROUP BY` interacts with `ORDER BY`.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **GROUP BY** | Splits the rows into groups that share a value. |
| **Group** | All rows that have the same value in the grouped column. |

#### The Idea in Plain Words

`GROUP BY` is like sorting laundry into separate baskets by colour. Each basket is a group. Then you run the aggregate on each basket. One row per group in the result.

#### See It in Code

Before you run this, guess the number of rows in the result.

In [ ]:
result = run_query("""
    SELECT strftime('%m', date(GMT)) AS month,
           COUNT(*) AS days,
           SUM(Precipitationmm) AS total_rain
    FROM london_2014
    GROUP BY month
    ORDER BY month
""")
print(result)




**Expected output:**
```text
  month  days  total_rain
0    01    10         4.5
1    06    10         1.5
2    12     5         4.5
```

**What happened:** Rows were grouped by month. Each group produced one row with its count and total rain.

#### Common Mistakes

```text
OperationalError: column "GMT" must appear in the GROUP BY clause
```

You selected a column that is not in `GROUP BY` and not aggregated. Fix: add it to `GROUP BY` or wrap it in an aggregate.

#### Quick Check
1. What does `GROUP BY` split rows by?
2. How many rows does `GROUP BY` produce per group?
3. What must a `SELECT` column be if it is not aggregated?

#### Lesson Recap
- `GROUP BY` splits rows into groups.
- One row per group in the result.
- Other columns must be aggregated or grouped.

---

### Lesson 4.3: HAVING

#### What You Will Learn
- Why `WHERE` cannot filter aggregates.
- How `HAVING` filters after grouping.
- When to use `WHERE` and when to use `HAVING`.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **HAVING** | Filters groups after `GROUP BY`. |

#### The Idea in Plain Words

`WHERE` runs on each row before grouping. `HAVING` runs on each group after grouping. If you want "only rows where it rained", use `WHERE`. If you want "only months where total rain was over 3 mm", use `HAVING`.

#### See It in Code

Before you run this, guess which months survive.

In [ ]:
result = run_query("""
    SELECT strftime('%m', date(GMT)) AS month,
           SUM(Precipitationmm) AS total_rain
    FROM london_2014
    GROUP BY month
    HAVING SUM(Precipitationmm) > 3
    ORDER BY month
""")
print(result)




**Expected output:**
```text
  month  total_rain
0    01         4.5
1    12         4.5
```

**What happened:** Only January and December had total rain above 3 mm. June was dropped.

#### Common Mistakes

```text
OperationalError: misuse of aggregate in WHERE clause
```

You put `SUM` in `WHERE`. Fix: use `HAVING`.

#### Quick Check
1. Does `WHERE` run before or after `GROUP BY`?
2. Does `HAVING` run before or after `GROUP BY`?
3. Why can `HAVING SUM(...)` work but `WHERE SUM(...)` fail?

#### Lesson Recap
- `WHERE` filters rows.
- `HAVING` filters groups.
- Aggregates in `HAVING`, not in `WHERE`.

---

### Lesson 4.4: ROUND

#### What You Will Learn
- How `ROUND` trims decimal places.
- How to round an aggregate result.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **ROUND** | Reduces a number to a fixed number of decimal places. |

#### The Idea in Plain Words

A price of 12.3456 is hard to read. `ROUND(12.3456, 2)` gives 12.35. The second number is how many digits to keep after the decimal point.

#### See It in Code

Before you run this, guess the rounded result.

In [ ]:
result = run_query("""
    SELECT ROUND(AVG(Precipitationmm), 2) AS avg_rain,
           ROUND(SUM(Precipitationmm), 2) AS total_rain
    FROM london_2014
""")
print(result)




**Expected output:**
```text
   avg_rain  total_rain
0      0.42        10.5
```

**What happened:** Both aggregates were rounded to two decimal places.

#### Common Mistakes

```text
OperationalError: near "(": syntax error
```

You missed a comma or parenthesis. Fix: `ROUND(value, places)` has two arguments.

#### Quick Check
1. What does `ROUND(x, 2)` do?
2. Does `ROUND` change the stored value or only the output?

#### Lesson Recap
- `ROUND` keeps a fixed number of decimals.
- It only changes the output, not the stored value.

### Phase 4 Summary
- `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` collapse rows.
- `GROUP BY` splits rows into groups.
- `HAVING` filters groups.
- `ROUND` trims decimals.

### Phase 4 Practice Task: GDP by country shape

**Difficulty:** Medium | **Time:** 25 minutes

**Skills used:**
- From this phase: `COUNT`, `SUM`, `AVG` (Lesson 4.1), `GROUP BY` (Lesson 4.2), `HAVING` (Lesson 4.3), `ROUND` (Lesson 4.4)
- From earlier phases: `SELECT`, `ORDER BY` (Phase 1)

**The task:** Write one SQL query on `wb_gdp_2013`. Return one row per country (each row is its own group). Show three columns: `country`, `gdp_bn` (GDP in billions, rounded to 2 decimals), and `gdp_usd` (the raw GDP). Sort by `gdp_usd` descending. Keep only rows where `gdp_usd` is over 20 billion.

The rows in `wb_gdp_2013`:

| country | NY.GDP.MKTP.CD |
|---|---|
| Afghanistan | 20458940000 |
| Albania | 12781030000 |
| Algeria | 209703500000 |
| Angola | 138356800000 |
| India | 1876787989000 |

**Expected output:**
```text
       country  gdp_bn     gdp_usd
0        India  1876.79  1.876788e+12
1      Algeria   209.70  2.097035e+11
2       Angola   138.36  1.383568e+11
3  Afghanistan    20.46  2.045894e+10
```

---

## Phase 5: Joining Tables

**Phase goal:** Combine two or more tables on a shared column.

**You need before this phase:** Phase 4.

#### Setup Cell

In [ ]:
# Check the two tables side by side.
print(run_query("SELECT * FROM wb_gdp_2013"))
print(run_query("SELECT * FROM wb_le_2013"))




### Lesson 5.1: INNER JOIN

#### What You Will Learn
- What a join is.
- How to write `JOIN ... ON`.
- Why an inner join drops rows with no match.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **JOIN** | Combine rows from two tables. |
| **ON** | The condition that decides which rows match. |
| **INNER JOIN** | Keep only rows that match in both tables. |
| **Alias** | A short name for a table, like `g` for `wb_gdp_2013`. |

#### The Idea in Plain Words

Two guest lists. One has people bringing food. The other has people bringing drinks. An inner join keeps only the people on both lists.

You write `FROM table_a a JOIN table_b b ON a.key = b.key`. The `a` and `b` are short aliases so you can write `a.country` instead of the full name.

#### See It in Code

Before you run this, guess how many rows survive.

In [ ]:
result = run_query("""
    SELECT g.country,
           g.[NY.GDP.MKTP.CD] AS gdp,
           l.[SP.DYN.LE00.IN] AS life
    FROM wb_gdp_2013 g
    JOIN wb_le_2013 l ON g.country = l.country
""")
print(result)




**Expected output:**
```text
       country           gdp  life
0  Afghanistan  2.045894e+10  60.3
1      Albania  1.278103e+10  77.8
2      Algeria  2.097035e+11  75.1
3       Angola  1.383568e+11  52.4
```

**What happened:** India (in GDP only) and Nigeria (in life only) were dropped. Only the four countries in both tables survived.

#### Common Mistakes

```text
OperationalError: ambiguous column name: country
```

Both tables have a `country` column and you wrote it without a prefix. Fix: write `g.country` or `l.country`.

#### Quick Check
1. What does `JOIN ... ON` do?
2. Why did India and Nigeria disappear?
3. What is an alias, and why use one?

#### Lesson Recap
- `JOIN ... ON` combines rows from two tables.
- An inner join keeps only matches.
- Aliases keep queries short.

---

### Lesson 5.2: LEFT JOIN and finding non-matches

#### What You Will Learn
- How a `LEFT JOIN` keeps all rows of the left table.
- How to find rows that have no match with `IS NULL`.
- How to count the rows a join would lose.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **LEFT JOIN** | Keep all rows of the left table, add matches from the right. |
| **NULL** | A missing value. |
| **IS NULL** | Test whether a value is missing. |
| **IS NOT NULL** | Test whether a value exists. |

#### The Idea in Plain Words

A left join is a guest list of the left table. Everyone on the left appears in the result. If they are also on the right list, their details are filled in. If not, their right-hand columns are NULL.

`IS NULL` tests for a blank. A blank is not the same as zero or the empty string.

#### See It in Code

Before you run this, guess the count.

In [ ]:
result = run_query("""
    SELECT COUNT(*) AS rows_lost
    FROM wb_gdp_2013 g
    LEFT JOIN wb_le_2013 l ON g.country = l.country
    WHERE l.[SP.DYN.LE00.IN] IS NULL
""")
print(result)




**Expected output:**
```text
   rows_lost
0          1
```

**What happened:** One country (India) had a GDP row but no matching life-expectancy row. Its `life` value was NULL after the left join.

#### Common Mistakes

```text
OperatorError: near "= NULL": syntax error
```

You wrote `= NULL`. Fix: write `IS NULL`. NULL is never equal to anything, not even itself.

#### Quick Check
1. What does `LEFT JOIN` keep?
2. Why does `= NULL` not work?
3. Which row was counted as lost?

#### Lesson Recap
- `LEFT JOIN` keeps all rows of the left table.
- `IS NULL` finds blanks.
- `= NULL` never matches; use `IS NULL`.

---

### Lesson 5.3: Joining three tables

#### What You Will Learn
- How to chain a second `JOIN`.
- How the order of joins changes nothing about the result but affects readability.
- How to combine joins with aggregates.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **Chained join** | Two or more `JOIN` clauses in one query. |

#### The Idea in Plain Words

Three guest lists, three skills: food, drinks, music. A three-way inner join keeps only the people on all three lists. Each `JOIN` adds another table on a shared key.

#### See It in Code

Before you run this, guess how many rows survive.

In [ ]:
result = run_query("""
    SELECT g.country,
           g.[NY.GDP.MKTP.CD] AS gdp,
           l.[SP.DYN.LE00.IN] AS life,
           p.[SP.POP.TOTL] AS population
    FROM wb_gdp_2013 g
    JOIN wb_le_2013 l ON g.country = l.country
    JOIN wb_pop_2013 p ON g.country = p.country
""")
print(result)




**Expected output:**
```text
       country           gdp  life  population
0  Afghanistan  2.045894e+10  60.3  30552000.0
1      Albania  1.278103e+10  77.8   2895000.0
2      Algeria  2.097035e+11  75.1  38282000.0
3       Angola  1.383568e+11  52.4  25789000.0
```

**What happened:** Only countries in all three tables survived. India and Nigeria were dropped because they are not in `wb_pop_2013`.

#### Common Mistakes

```text
OperationalError: no such column: g.country
```

You forgot the alias `g`. Fix: include `wb_gdp_2013 g` after `FROM`.

#### Quick Check
1. How many `JOIN` clauses does a three-way join have?
2. What is dropped by an inner join across three tables?
3. Does the order of `JOIN` clauses change the result?

#### Lesson Recap
- Chain `JOIN` clauses to combine three or more tables.
- An inner join across three tables keeps only rows in all three.
- The order of joins does not change the result.

### Phase 5 Summary
- `JOIN ... ON` combines tables on a shared key.
- `LEFT JOIN` keeps all rows of the left table.
- `IS NULL` finds rows with no match.
- You can chain two or more joins.

### Phase 5 Practice Task: GDP, life, and population

**Difficulty:** Medium-Hard | **Time:** 35 minutes

**Skills used:**
- From this phase: `INNER JOIN` (Lesson 5.1), `LEFT JOIN` and `IS NULL` (Lesson 5.2), three-way join (Lesson 5.3)
- From earlier phases: `WHERE`, `ORDER BY`, `ROUND` (Phase 1 and Phase 4)

**The task:** Write two queries.

1. Join `wb_gdp_2013`, `wb_le_2013`, and `wb_pop_2013` on `country` with inner joins. Return `country`, `gdp_bn` (GDP in billions, rounded to 2 decimals), `life`, and `population_m` (population in millions, rounded to 2 decimals). Sort by `gdp_bn` descending.
2. Using a left join from `wb_gdp_2013` to `wb_le_2013`, count how many GDP rows have no matching life-expectancy value.

The data:

| country | NY.GDP.MKTP.CD | SP.DYN.LE00.IN | SP.POP.TOTL |
|---|---|---|---|
| Afghanistan | 20458940000 | 60.3 | 30552000 |
| Albania | 12781030000 | 77.8 | 2895000 |
| Algeria | 209703500000 | 75.1 | 38282000 |
| Angola | 138356800000 | 52.4 | 25789000 |
| India | 1876787989000 | (none) | (none) |
| Nigeria | (none) | 55.0 | (none) |

**Expected output:**
```text
Query 1:
       country  gdp_bn  life  population_m
0      Algeria  209.70  75.1         38.28
1       Angola  138.36  52.4         25.79
2  Afghanistan   20.46  60.3         30.55
3      Albania   12.78  77.8          2.90

Query 2:
   rows_lost
0          1
```

---

## Phase 6: CTEs and Window Functions

**Phase goal:** Wrap queries in named sub-queries and compute running totals with window functions.

**You need before this phase:** Phase 5.

#### Setup Cell

In [ ]:
# A quick sanity check that the weather table still has 25 rows.
result = run_query("SELECT COUNT(*) AS n FROM london_2014")
print(result)




### Lesson 6.1: WITH (the CTE)

#### What You Will Learn
- What a CTE is.
- How to write `WITH name AS (...)`.
- How a CTE lets you filter on a result.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **CTE** | A named sub-query that you can use later in the same query. |
| **WITH** | The SQL keyword that starts a CTE. |

#### The Idea in Plain Words

A CTE is like naming a recipe step. "First, boil the water. Then, add the pasta to the boiled water." You name the first step and reuse it in the second.

`WITH daily AS (SELECT ...) SELECT ... FROM daily` names a query `daily` and then queries it. The CTE is not saved; it lives only for this one query.

#### See It in Code

Before you run this, guess the count.

In [ ]:
result = run_query("""
    WITH january AS (
        SELECT GMT, [Max TemperatureC]
        FROM london_2014
        WHERE GMT BETWEEN '2014-01-01' AND '2014-01-31'
    )
    SELECT COUNT(*) AS january_days
    FROM january
""")
print(result)




**Expected output:**
```text
   january_days
0            10
```

**What happened:** The CTE `january` held only January rows. The outer query counted them.

#### Common Mistakes

```text
OperationalError: near "WITH": syntax error
```

You put a semicolon after the CTE. Fix: `WITH name AS (...) SELECT ...` is one statement, no semicolon in the middle.

#### Quick Check
1. What does `WITH` do?
2. How long does a CTE live?
3. Why filter in a CTE instead of `WHERE` at the end?

#### Lesson Recap
- `WITH name AS (...)` names a sub-query.
- The CTE lives only for one statement.
- A CTE lets you filter a result in a second step.

---

### Lesson 6.2: SUM OVER and ROWS BETWEEN

#### What You Will Learn
- What a window function is.
- How `SUM(x) OVER (...)` slides over rows.
- How `ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING` defines a 7-row window.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **Window function** | A calculation that sees many rows and returns one value per row. |
| **OVER** | The keyword that starts the window. |
| **ORDER BY** (in a window) | The order the window slides in. |
| **ROWS BETWEEN** | The start and end of the window. |
| **CURRENT ROW** | The row being calculated. |
| **n FOLLOWING** | The n rows after the current row. |

#### The Idea in Plain Words

A normal `SUM` returns one total for the whole table. A window `SUM` returns one value per row, looking at a small window of neighbours.

`SUM(x) OVER (ORDER BY day ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING)` for row 3 sums rows 3, 4, 5, 6, 7, 8, 9. That is a 7-row rolling total.

#### See It in Code

Before you run this, guess the first total.

In [ ]:
result = run_query("""
    SELECT GMT,
           Precipitationmm,
           SUM(Precipitationmm) OVER (
               ORDER BY GMT
               ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
           ) AS rain_7d,
           COUNT(*) OVER (
               ORDER BY GMT
               ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
           ) AS days_in_window
    FROM london_2014
    WHERE GMT BETWEEN '2014-01-01' AND '2014-01-10'
    ORDER BY GMT
""")
print(result)




**Expected output:**
```text
          GMT  Precipitationmm  rain_7d  days_in_window
0  2014-01-01              0.0      4.0               7
1  2014-01-02              2.5      4.0               7
2  2014-01-03              0.0      1.5               7
3  2014-01-04              1.0      2.0               7
4  2014-01-05              0.0      1.0               6
5  2014-01-06              0.5      0.5               5
6  2014-01-07              0.0      0.0               4
7  2014-01-08              0.0      0.0               3
8  2014-01-09              0.0      0.0               2
9  2014-01-10              0.5      0.5               1
```

**What happened:** Row 1 looked at rows 1 to 7 (7 days). Row 4 looked at rows 4 to 10 (7 days). Rows 5 to 10 had fewer than 7 days ahead, so the window shrinks to whatever is left.

#### Common Mistakes

```text
OperationalError: near "FOLLOWING": syntax error
```

You wrote `FOLLOWING` without a number. Fix: `6 FOLLOWING`.

#### Quick Check
1. What does `SUM(x) OVER (...)` return per row?
2. What does `CURRENT ROW AND 6 FOLLOWING` mean?
3. Why does `days_in_window` shrink at the end?

#### Lesson Recap
- `OVER (...)` turns `SUM` into a window function.
- `ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING` slides a 7-row window.
- The window shrinks near the end of the range.

---

### Lesson 6.3: A rolling 7-day total

#### What You Will Learn
- How to combine a CTE with a window function.
- How to keep only rows whose window has 7 days.
- How to find the driest 7-day period.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **Rolling total** | A total computed over a sliding window of recent rows. |
| **Full window** | A window with the expected number of rows. |

#### The Idea in Plain Words

You want the driest 7-day holiday. You compute a 7-day rolling rain total for every start date. You keep only starts with a full 7 days. You sort by total rain, smallest first. The top row is the answer.

#### See It in Code

Before you run this, guess the start day.

In [ ]:
result = run_query("""
    WITH daily AS (
        SELECT GMT, Precipitationmm
        FROM london_2014
        WHERE GMT BETWEEN '2014-06-01' AND '2014-08-31'
    ),
    windows AS (
        SELECT GMT AS start_day,
               SUM(Precipitationmm) OVER (
                   ORDER BY GMT
                   ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
               ) AS rain_7d,
               COUNT(*) OVER (
                   ORDER BY GMT
                   ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
               ) AS days_in_window
        FROM daily
    )
    SELECT start_day, ROUND(rain_7d, 2) AS rain_mm_7d
    FROM windows
    WHERE days_in_window = 7
    ORDER BY rain_7d ASC, start_day ASC
    LIMIT 1
""")
print(result)




**Expected output:**
```text
    start_day  rain_mm_7d
0  2014-06-01         0.5
```

**What happened:** Only June 1 had a full 7-day window and the least rain (0.5 mm).

#### Common Mistakes

```text
OperationalError: no such column: days_in_window
```

You tried to use `days_in_window` inside the same `SELECT`. Fix: wrap in a CTE first.

#### Quick Check
1. Why filter `days_in_window = 7`?
2. Why sort by `rain_7d ASC`?
3. Why filter the CTE to June through August only?

#### Lesson Recap
- A CTE can hold the window function result.
- Filter to a full window.
- Sort ascending and use `LIMIT` to pick the top row.

---

### Lesson 6.4: ROW_NUMBER and RANK

#### What You Will Learn
- How `ROW_NUMBER` numbers rows.
- How `RANK` handles ties.
- How `PARTITION BY` restarts numbering per group.

#### Key Terms
| Term | Simple meaning |
|---|---|
| **ROW_NUMBER** | Numbers rows from 1 in a given order. |
| **RANK** | Numbers rows but gives equal rank to ties, skipping numbers. |
| **DENSE_RANK** | Like `RANK` but no gaps after ties. |
| **PARTITION BY** | Splits the window into groups. |

#### The Idea in Plain Words

`ROW_NUMBER` is a referee counting every runner in a race. `RANK` gives first place, second place, and so on. Two runners tied for second both get rank 2, and the next gets 4. `PARTITION BY` is like running separate races per age group.

#### See It in Code

Before you run this, guess the top three rows.

In [ ]:
result = run_query("""
    SELECT country,
           [SP.POP.TOTL] AS population,
           RANK() OVER (ORDER BY [SP.POP.TOTL] DESC) AS pop_rank
    FROM wb_pop_2013
""")
print(result)




**Expected output:**
```text
       country  population  pop_rank
0      Algeria  38282000.0         1
1  Afghanistan  30552000.0         2
2       Angola  25789000.0         3
3      Albania   2895000.0         4
```

**What happened:** `RANK` ordered countries by population and numbered them. No ties, so no shared ranks.

#### Common Mistakes

```text
OperationalError: no such function: RANK
```

Your SQLite version is too old. Fix: upgrade SQLite to 3.25 or later.

#### Quick Check
1. What does `RANK` give to ties?
2. What does `PARTITION BY` do?
3. Which function numbers rows without ties?

#### Lesson Recap
- `ROW_NUMBER`, `RANK`, `DENSE_RANK` number rows.
- `PARTITION BY` splits the window into groups.
- All three are window functions.

### Phase 6 Summary
- `WITH` creates a named sub-query.
- Window functions slide over rows.
- `ROWS BETWEEN` sets the window size.
- `RANK` and `ROW_NUMBER` number rows.

### Phase 6 Practice Task: Driest summer week

**Difficulty:** Hard | **Time:** 40 minutes

**Skills used:**
- From this phase: CTE (Lesson 6.1), `SUM OVER ROWS BETWEEN` (Lesson 6.2), rolling total (Lesson 6.3), `RANK` (Lesson 6.4)
- From earlier phases: `BETWEEN` (Phase 3), `COUNT`, `ROUND` (Phase 4)

**The task:** The table `london_2014` has the following summer rows (June 1 to June 10):

| GMT | Precipitationmm |
|---|---|
| 2014-06-01 | 0.0 |
| 2014-06-02 | 0.0 |
| 2014-06-03 | 0.0 |
| 2014-06-04 | 0.5 |
| 2014-06-05 | 0.0 |
| 2014-06-06 | 0.0 |
| 2014-06-07 | 0.0 |
| 2014-06-08 | 1.0 |
| 2014-06-09 | 0.0 |
| 2014-06-10 | 0.0 |

Find the driest 7-day period. Return one row with two columns: `start_day` (a text date) and `rain_mm_7d` (total rain over that period, rounded to 2 decimals). If several periods tie, pick the earliest. Use a CTE and a window function.

**Expected output:**
```text
    start_day  rain_mm_7d
0  2014-06-01         0.5
```

---

## Glossary

- **Aggregate function:** A function that collapses many rows into one value.
- **Alias:** A short name for a column or table.
- **AND:** A keyword meaning both conditions must be true.
- **AS:** A keyword that renames a column in the output.
- **ASC:** Sort order from smallest to largest (the default).
- **AVG:** The average of a numeric column.
- **BETWEEN:** Keep values inside a range, both ends included.
- **Bracket-quote:** Wrap a name in `[ ... ]` so SQL reads it as one name.
- **CAST:** Change a value's type.
- **Chained join:** Two or more `JOIN` clauses in one query.
- **Column:** One field that every row of a table shares.
- **Condition:** A test that is true or false for each row.
- **Conversion:** Changing a value from one type to another.
- **COUNT:** The number of rows.
- **CTE:** A named sub-query used inside a larger query.
- **CURRENT ROW:** The row the window function is computing for.
- **Date type:** A real date value, not text.
- **DAY:** The day part of a date.
- **DENSE_RANK:** Numbers rows, gives ties equal rank, no gaps.
- **DESC:** Sort order from largest to smallest.
- **DISTINCT:** Removes duplicate rows from the result.
- **FOLLOWING:** The rows after the current row in a window.
- **FROM:** The SQL keyword that names the table.
- **Full window:** A window with the expected number of rows.
- **GROUP BY:** Splits rows into groups.
- **Group:** All rows that share a value in the grouped column.
- **HAVING:** Filters groups after `GROUP BY`.
- **Identifier:** The name of a table or column.
- **INNER JOIN:** Keep only rows that match in both tables.
- **INTEGER:** Whole numbers.
- **IS NULL:** Test whether a value is missing.
- **ISO date:** A date written as `YYYY-MM-DD`.
- **JOIN:** Combine rows from two tables.
- **LEFT JOIN:** Keep all rows of the left table.
- **LIMIT:** Keep the first N rows after sorting.
- **LOWER:** Turn text into lowercase.
- **LTRIM:** Remove spaces from the left end.
- **MAX:** The largest value.
- **MIN:** The smallest value.
- **MONTH:** The month part of a date.
- **NULL:** A missing value.
- **ON:** The condition that decides which rows match.
- **OR:** A keyword meaning at least one condition must be true.
- **ORDER BY:** Sort the result.
- **OVER:** The keyword that starts a window function.
- **PARTITION BY:** Splits the window into groups.
- **Query:** One request written in SQL.
- **RANK:** Numbers rows, gives ties equal rank, skips numbers.
- **REAL:** Decimal numbers.
- **REPLACE:** Swap one piece of text for another.
- **Reserved word:** A word SQL treats specially, like `SELECT`.
- **ROLLING TOTAL:** A total computed over a sliding window.
- **ROUND:** Reduce a number to a fixed number of decimal places.
- **ROW:** One record in a table.
- **ROWS BETWEEN:** The start and end of a window.
- **ROW_NUMBER:** Numbers rows from 1 in a given order.
- **RTRIM:** Remove spaces from the right end.
- **SELECT:** The SQL keyword that names the columns.
- **SQL:** The language used to ask a database for data.
- **strftime:** Format a date into text (SQLite).
- **SUM:** The total of a numeric column.
- **Table:** A grid of rows and columns stored in a database.
- **TEXT:** Text values.
- **TOP:** The SQL Server version of `LIMIT`.
- **TRIM:** Remove spaces from both ends of text.
- **TRY_CAST:** Change a value's type, return NULL on failure.
- **TRY_CONVERT:** Convert text to a date, return NULL on failure (SQL Server).
- **UPPER:** Turn text into uppercase.
- **WHERE:** Keep only rows that match a condition.
- **Window function:** A calculation that sees many rows and returns one value per row.
- **WITH:** The SQL keyword that starts a CTE.
- **YEAR:** The year part of a date.
- **<>:** "Not equal to".

---

## Answer Key

### Phase 1
**Lesson 1.1**
1. `SELECT *` means "every column".
2. `FROM` names the table.
3. The name has a space, so it must be wrapped in square brackets.

**Lesson 1.2**
1. Rows where the maximum temperature is 10 or higher.
2. `AND` requires both conditions to be true.
3. `<>` (some databases also accept `!=`).

**Lesson 1.3**
1. Largest first.
2. At the very end of the query, after `ORDER BY`.
3. Right after `SELECT`: `SELECT TOP 5 ...`.

**Lesson 1.4**
1. Gives a column a new name in the output.
2. Duplicate rows.
3. A nickname for a column or table.

### Phase 2
**Lesson 2.1**
1. When the name contains a space, a dot, or a symbol like `<`.
2. It has a special meaning to SQL and cannot be used as a plain name.
3. Only the way you type it. The stored name is the same.

**Lesson 2.2**
1. Deletes every `<br />` from the text.
2. The number of rows.
3. So the comparison `> 350` compares numbers, not text.

**Lesson 2.3**
1. The integer 186.
2. It returns 186 in SQLite because SQLite reads leading digits and stops at the non-numeric text. SQL Server would raise an error.
3. It returns NULL on bad data instead of stopping the whole query.

**Lesson 2.4**
1. Spaces at both ends.
2. `LTRIM`.
3. `UPPER`.

### Phase 3
**Lesson 3.1**
1. International Organization for Standardization.
2. ISO dates are padded so they sort as text exactly as dates would.
3. Yes, both ends.

**Lesson 3.2**
1. The same date as a real date value.
2. Because ISO text sorts as dates.
3. `TRY_CONVERT(date, GMT)`.

**Lesson 3.3**
1. `'%m'`.
2. `YEAR(date)`.
3. Text, not a number, by default.

### Phase 4
**Lesson 4.1**
1. The number of rows in the table.
2. `MIN`.
3. No. `AVG` skips NULL values.

**Lesson 4.2**
1. By the value of one or more columns.
2. One row per group.
3. It must appear in `GROUP BY` or be wrapped in an aggregate.

**Lesson 4.3**
1. Before.
2. After.
3. `WHERE` runs before groups exist, so an aggregate has nothing to work on.

**Lesson 4.4**
1. Keeps two digits after the decimal point.
2. Only the output. The stored value is unchanged.

### Phase 5
**Lesson 5.1**
1. Combines rows from two tables where the `ON` condition is true.
2. They were missing from one of the two tables.
3. A short name for a table or column. It keeps queries short.

**Lesson 5.2**
1. All rows of the left table.
2. NULL is not equal to anything, not even itself. Use `IS NULL`.
3. India (it has GDP but no life-expectancy row).

**Lesson 5.3**
1. Two `JOIN` clauses for three tables.
2. Any row missing from any of the tables.
3. No, the result is the same.

### Phase 6
**Lesson 6.1**
1. Names a sub-query.
2. For one statement only.
3. So you can filter on the CTE's result in a second step.

**Lesson 6.2**
1. One value per row, computed from a window of neighbours.
2. The current row plus the six rows after it.
3. Because near the end of the range there are fewer than seven rows ahead.

**Lesson 6.3**
1. To exclude partial windows at the end of the range.
2. Smallest rain first, so the driest week is on top.
3. Summer is June, July, and August. Filtering keeps the window contiguous.

**Lesson 6.4**
1. The same rank for both tied rows.
2. Splits the window into groups.
3. `ROW_NUMBER`.